In [1]:

from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

In [2]:

load_dotenv()

False

In [3]:
# 2. Create Hugging Face LLM
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-R1-0528",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [4]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [6]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [7]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [8]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])

<think>
Okay, the user is asking about the distance from the Earth to the Moon. Let me recall the exact figures. The average distance is about 384,400 kilometers. But I should also mention that it's not a constant distance because the Moon's orbit is elliptical. So, the perigee and apogee distances are important too—around 362,600 km at closest and 405,400 km at farthest. 

Hmm, the user might be a student working on homework or just someone curious about space. Either way, providing both average and variable distances gives a complete picture. Maybe they need precise numbers for a project, or perhaps they're just casually interested. Either way, including the variation shows the dynamic nature of orbital mechanics. 

I should also explain why the distance changes—because of the elliptical orbit. That adds context without being too technical. And converting to miles could be helpful since some users might be more familiar with that unit. Let's see... 238,855 miles average, 225,623 mile

In [9]:
model.invoke('How far is moon from the earth?').content

"<think>\nOkay, the user is asking about the distance from the Earth to the Moon. Let me recall the exact figures. The average distance is about 384,400 kilometers. But I should mention that it's not constant because the Moon's orbit is elliptical. So, the closest point is perigee, around 363,000 km, and the farthest is apogee, about 405,000 km.  \n\nHmm, maybe the user is curious why the distance varies. Should I explain the elliptical orbit? Also, they might not realize how precise measurements are now. Including that we use laser reflectors left by Apollo missions could add interesting context.  \n\nWait, why is the user asking this? Could be for schoolwork, general knowledge, or planning an event like stargazing. Since they didn't specify, keeping it straightforward but including key points like variation and measurement methods covers possible needs.  \n\nAlso, converting to miles might help since some users might be more familiar with that. So, adding both kilometers and miles co